# 🚗 Dehazing and YOLOv8


---

## 🎯 **Objective**
To build a lightweight, real-time system that:
- Enhances foggy dashcam footage using **CLAHE-based dehazing**
- Detects objects like **cars, buses, trucks, potholes** using **YOLOv8**
- Compares detection accuracy between original and dehazed frames
- Aims to improve safety in autonomous and surveillance systems under poor visibility

---

## 🔧 **Pipeline Overview**

1. **Video Input** — Load real-world foggy dashcam footage
2. **Dehazing with CLAHE** — Enhance image contrast to improve visibility
3. **Object Detection** — Use YOLOv8 for detecting target classes
4. **Side-by-Side Comparison** — Show detections on both original & dehazed frames
5. **Detection Count Analysis** — Count & compare detections frame-by-frame

---

## 📊 Project Pipeline




---

## 🔧 Techniques Used

- **CLAHE (Contrast Limited Adaptive Histogram Equalization)** for dehazing
- **YOLOv8** pre-trained model (via Ultralytics) for object detection
- **OpenCV + Matplotlib** for video frame handling and visualization

---

## ✅ Evaluation Metric

- **Average Detections per Frame** on both:
  - Original video
  - Dehazed video

## Import Libraries

In [1]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Install YOLOv8 package (only once in Colab/Jupyter)
# !pip install ultralytics

# Import required modules
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


📘 This cell sets up the core libraries needed for image processing, plotting, and YOLO inference.

## Load Model & Target Labels

In [2]:
# Load pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Only detect these relevant road objects
TARGET_CLASSES = ['car', 'bus', 'truck', 'pothole']

📘 Loads YOLOv8 and selects classes we want to focus on detecting (others ignored).

## Dehazing with CLAHE

In [3]:
def clahe_dehaze(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def gamma_correction_dehaze(image, gamma=1.5):
    # Build a lookup table mapping pixel values [0, 255] to adjusted gamma values
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255 for i in np.arange(256)]).astype("uint8")
    
    # Apply gamma correction
    return cv2.LUT(image, table)

def adjust_brightness_contrast(image, alpha=1.3, beta=30):
    """
    alpha: Contrast control (1.0-3.0)
    beta: Brightness control (0-100)
    """
    adjusted = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
    return adjusted

def simple_hist_equalization(image):
    channels = cv2.split(image)
    eq_channels = [cv2.equalizeHist(ch) for ch in channels]
    eq_image = cv2.merge(eq_channels)
    return eq_image

def basic_dark_channel(image, size=15):
    """Apply a very basic version of Dark Channel Prior"""
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    dark = cv2.erode(min_channel, kernel)
    norm_image = cv2.normalize(dark, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.merge([norm_image]*3)




## general dehaze() function:

In [4]:
# ==== Dehazing selector ====
DEHAZE_METHOD = "clahe"  # Choose: "clahe", "gamma", "brightness", "hist_eq", "dark_channel"

def dehaze(image):
    if DEHAZE_METHOD == "clahe":
        return clahe_dehaze(image)
    elif DEHAZE_METHOD == "gamma":
        return gamma_correction_dehaze(image)
    elif DEHAZE_METHOD == "brightness":
        return adjust_brightness_contrast(image)
    elif DEHAZE_METHOD == "hist_eq":
        return simple_hist_equalization(image)
    elif DEHAZE_METHOD == "dark_channel":
        return basic_dark_channel(image)
    else:
        raise ValueError(f"Unknown dehazing method: {DEHAZE_METHOD}")


📘 Improves frame contrast using histogram equalization in LAB color space, simulating fog removal.

## Object Detection and Visualization

In [5]:
def detect_and_draw(image, label_filter):
    results = model(image, verbose=False)
    detections = 0
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            label = result.names[int(box.cls[0])]
            conf = box.conf[0].item()

            if label in label_filter:
                detections += 1
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(image, f"{label} {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    return image, detections


In [6]:
def process_all_videos(video_folder, max_frames=20, save_visuals=False, visuals_dir="visuals"):
    results = []

    if save_visuals and not os.path.exists(visuals_dir):
        os.makedirs(visuals_dir)

    video_files = [f for f in os.listdir(video_folder) if f.endswith(('.mp4', '.avi', '.mov'))]

    for i, video_file in enumerate(video_files):
        print(f"[{i+1}/{len(video_files)}] Processing: {video_file}")
        video_path = os.path.join(video_folder, video_file)
        
        # Run your main pipeline on this video
        raw_detections_per_frame, dehazed_detections_per_frame = main(video_path, max_frames=max_frames)

        results.append({
            "Video": video_file,
            "Raw Detections/Frame": raw_detections_per_frame,
            "Dehazed Detections/Frame": dehazed_detections_per_frame
        })

        # Optionally save side-by-side visuals for first few videos
        if save_visuals and i < 5:
            cap = cv2.VideoCapture(video_path)
            ret, frame = cap.read()
            if ret:
                dehazed = dehaze(frame.copy())
                frame_raw, _ = detect_and_draw(frame.copy(), TARGET_CLASSES)
                frame_dehazed, _ = detect_and_draw(dehazed.copy(), TARGET_CLASSES)
                side_by_side = cv2.hconcat([frame_raw, frame_dehazed])
                save_path = os.path.join(visuals_dir, f"{os.path.splitext(video_file)[0]}_visual.png")
                cv2.imwrite(save_path, side_by_side)
            cap.release()

    # Convert to DataFrame
    df = pd.DataFrame(results)

    # Calculate detection improvement %
    df["Detection Gain (%)"] = df.apply(
        lambda row: round(((row["Dehazed Detections/Frame"] - row["Raw Detections/Frame"]) /
                           row["Raw Detections/Frame"]) * 100, 2) if row["Raw Detections/Frame"] != 0 else 0,
        axis=1
    )

    # Compute averages
    avg_raw = df["Raw Detections/Frame"].mean()
    avg_dehazed = df["Dehazed Detections/Frame"].mean()
    avg_gain = ((avg_dehazed - avg_raw) / avg_raw) * 100 if avg_raw != 0 else 0

    # Print summary
    print(f"\n--- Summary ---")
    print(f"Average Raw Detections/Frame: {avg_raw:.2f}")
    print(f"Average Dehazed Detections/Frame: {avg_dehazed:.2f}")
    print(f"Detection Gain (Dehazing): {avg_gain:.2f}%\n")

    # Save to CSV
    df.to_csv("dehazing_detection_comparison1.csv", index=False)

    return df

## Main Pipeline Function

In [7]:
def main(video_path, max_frames=20):
    cap = cv2.VideoCapture(video_path)
    frame_index = 0
    total_raw_detections = 0
    total_dehazed_detections = 0

    while cap.isOpened() and frame_index < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # dehazed = clahe_dehaze(frame.copy())
        dehazed = dehaze(frame.copy())


        frame_raw, count_raw = detect_and_draw(frame.copy(), TARGET_CLASSES)
        frame_dehazed, count_dehaze = detect_and_draw(dehazed.copy(), TARGET_CLASSES)

        total_raw_detections += count_raw
        total_dehazed_detections += count_dehaze

        # Convert to RGB for visualization
        raw_rgb = cv2.cvtColor(frame_raw, cv2.COLOR_BGR2RGB)
        dehazed_rgb = cv2.cvtColor(frame_dehazed, cv2.COLOR_BGR2RGB)

        # Show side-by-side
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(raw_rgb)
        plt.title(f"Original | Detections: {count_raw}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(dehazed_rgb)
        plt.title(f"Dehazed | Detections: {count_dehaze}")
        plt.axis('off')

        plt.tight_layout()
        display(plt.gcf())
        plt.close()

        frame_index += 1
        clear_output(wait=True)

    cap.release()

    # Calculate average detections per frame
    accuracy_raw = (total_raw_detections / max_frames) * 100 if max_frames else 0
    accuracy_dehazed = (total_dehazed_detections / max_frames) * 100 if max_frames else 0

    return round(accuracy_raw, 2), round(accuracy_dehazed, 2)



 Handles the entire video: reads frames, applies dehazing + detection, and shows side-by-side comparisons.

In [8]:
import os
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import pandas as pd

if __name__ == "__main__":
    video_folder_path = "dataset1"  
    df = process_all_videos(video_folder_path, max_frames=20, save_visuals=True)
    # plot_accuracy_comparison(df) 



--- Summary ---
Average Raw Detections/Frame: 564.64
Average Dehazed Detections/Frame: 578.09
Detection Gain (Dehazing): 2.38%



## Run and Show Accuracy

In [9]:
# Run the video through the pipeline
accuracy_orig, accuracy_dehazed = main("p1.mp4", max_frames=300)

# Print average detections per frame
print(f"📊 Original Video - Avg Detections/frame: {accuracy_orig:.2f}")
print(f"📊 Dehazed Video  - Avg Detections/frame: {accuracy_dehazed:.2f}")


📊 Original Video - Avg Detections/frame: 380.00
📊 Dehazed Video  - Avg Detections/frame: 385.67


## 📈 Outcome

We measure and compare the total number of valid detections across both versions to determine if **dehazing improves YOLOv8 detection accuracy**.

---
